# Do these models encode the same information?

**What you'll learn.** When several models embed the same entities, the useful
question is whether they *organise* them the same way. `embpy.tl` answers it with
representation-alignment metrics, and `alignment_matrix` gives you the whole
pairwise picture in one call.

The headline metric is **TSI** (Triplet Similarity Index): over triplets of
entities, how often do two representations agree on which of two neighbours is
closer? It has a property that makes it easy to read:

| TSI | meaning |
| --- | --- |
| 1.0 | identical ordering of every distance |
| ~0.5 | **the null** — unrelated representations |
| < 0.5 | systematically anti-correlated (rare) |

That fixed null is the point: unlike CKA, "low" has an absolute meaning.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd

from embpy import BioEmbedder, tl

embedder = BioEmbedder(device="auto", organism="human")

genes = ["TP53", "EGFR", "MYC", "BRCA1", "JUN", "STAT1", "IRF1", "CDK1",
         "GATA3", "FOXP3", "CD8A", "IL2", "CCND1", "RB1", "PTEN", "AKT1"]
base = ad.AnnData(
    X=np.zeros((len(genes), 1), dtype=np.float32),
    obs=pd.DataFrame({"symbol": genes}, index=genes),
)

## Embed the same entities with different model families

Three genuinely different views: a prior-knowledge table, a co-expression vector,
and a protein language model reading the amino-acid sequence.

In [ ]:
specs = [
    ("genept",   "gene",    "genept"),     # prior knowledge (text-derived)
    ("gene2vec", "gene",    "gene2vec"),   # co-expression
    ("esm2_8M",  "protein", "esm2"),       # protein language model
]

embeddings = {}
for model, entity, label in specs:
    out = embedder.embed(
        base.copy(), entity_type=entity, id_type="symbol", obs_column="symbol",
        model=model, output="anndata", key="X", attach_to="obs",
    )
    embeddings[label] = out.obsm["X"]

for label, mat in embeddings.items():
    print(f"{label:10s} {mat.shape}")

## The pairwise picture

`alignment_matrix` applies the metric to every pair. Feature dimensions may differ
(3072 vs 200 vs 320) — the metrics compare *geometry*, not coordinates.

In [ ]:
tsi_scores = tl.alignment_matrix(embeddings, metric="tsi")
tsi_scores.round(3)

Read these against the 0.5 null. Values in the high 0.5s / low 0.6s mean these
models agree only slightly more than chance about how these genes relate — a real
finding, and a warning that **your choice of model changes downstream results**.

## Why ship more than one metric

Different metrics disagree in informative ways. Compare CKA on the same data:

In [ ]:
cka_scores = tl.alignment_matrix(embeddings, metric="cka")
cka_scores.round(3)

Note they can rank the pairs **differently**. TSI is ordinal — it sees only which
distances are larger — while CKA compares inner-product structure and is dominated
by high-variance directions. Neither is "wrong"; they answer different questions.

You can also change the distance TSI uses, which matters for high-dimensional
biological data where cosine is often more meaningful than Euclidean:

In [ ]:
tl.alignment_matrix(embeddings, metric="tsi", distance="cosine").round(3)

## The property that matters for single-cell data

Real datasets contain a few extreme rows — dying cells, doublets, ambient-RNA
artefacts. CKA is a ratio of Frobenius norms, so those rows dominate it. Here we
take two representations that are *identical up to rotation* (true similarity 1.0)
and corrupt just 2% of rows:

In [ ]:
rng = np.random.default_rng(0)
n, d = 400, 32
X = rng.normal(size=(n, d))
Q, _ = np.linalg.qr(rng.normal(size=(d, d)))
Y = X @ Q                       # a pure rotation: true similarity is exactly 1.0

bad = rng.choice(n, size=int(0.02 * n), replace=False)
Y[bad] += rng.normal(scale=200.0, size=(len(bad), d))

print(f"TSI        : {tl.tsi(X, Y):.3f}   <- barely moves")
print(f"linear CKA : {tl.linear_cka(X, Y):.3f}   <- collapses")

CKA calls two rotation-identical representations unrelated. If you were sweeping
layers or ranking models with CKA, a handful of bad cells could hand you a
confidently wrong answer. This is the main reason to prefer the ordinal metrics
on single-cell data.

## Which layer should I cache?

The last layer of a foundation model is specialised toward its pretraining
objective, so it is routinely *not* the best for transfer. Embed at every layer,
then let `rank_layers` score them.

In [ ]:
layer_embeddings = {}
for layer in range(7):        # esm2_8M has 6 blocks + the embedding layer
    out = embedder.embed(
        base.copy(), entity_type="protein", id_type="symbol", obs_column="symbol",
        model="esm2_8M", layer=layer, output="anndata", key="X", attach_to="obs",
    )
    layer_embeddings[layer] = out.obsm["X"]

tl.rank_layers(layer_embeddings).round(3)

`to_final` says how much each layer already agrees with the final representation,
and `to_previous` how redundant it is with its neighbour. Pass `target=` (a
measurement you care about) to add a cross-validated `probe_score` column — then
the best layer is the one that predicts *your* label, not the last one by default.

> **Layer indexing.** These indices follow `extract_hidden_states`: index 0 is the
> embedding layer and index *b+1* is transformer block *b*. `extract_attention`
> uses a different convention (one entry per block, no embedding entry) — convert
> with `tl.block_to_hidden_state_index` / `tl.block_to_attention_index` before
> joining the two.

## The same call across single-cell foundation models

This is what the metrics were designed for: embed one common set of cells with
several foundation models and ask whether they encode the same biology. The call
is identical — only the models change. These need the heavier environments
(`scvi-tools`, `helical`), so it is shown rather than executed here:

```python
cells = ad.read_h5ad("my_cells.h5ad")

cell_embeddings = {}
for model in ["pca", "scvi", "geneformer", "scgpt"]:
    out = embedder.embed(cells.copy(), entity_type="cell", model=model,
                         output="anndata", key="X_emb")
    cell_embeddings[model] = out.obsm["X_emb"]

tl.alignment_matrix(cell_embeddings, metric="tsi", distance="cosine")
```

On more than a few thousand cells, `method="auto"` switches from the exact path to
a sampling estimator whose sample size is **independent of N** — 738 triplets give
you ±0.05 with 95% confidence whether you have 10³ or 10⁶ cells:

```python
tl.sample_size_for(epsilon=0.05, delta=0.05)   # 738
```

## Takeaway

- `alignment_matrix` turns "do these models agree?" into a labelled table.
- TSI has a **fixed 0.5 null**, so scores are comparable across datasets.
- TSI survives outliers that make CKA collapse — decisive for single-cell data.
- `rank_layers` picks a layer from evidence instead of defaulting to the last one.

**Next:** [Which model captures my biology?](04_benchmark_models.ipynb) turns this
structural comparison into a task-specific score.